# 03 — Results analysis (Chapter 5)

Builds every Chapter 5 table and figure from the committed pipeline outputs. All computation lives in
`src/results_analysis.py`; this notebook only calls it, shows the results and saves them to
`dissertation/tables/` and `dissertation/figures/` (interactive `.html`, plus `.png` when plotly's exporter is available).

**Prerequisites:** `make prepare`, `make sweep`, `make profile` — i.e. `data/processed/`, `experiments/results_raw.csv`
and `experiments/latency_results.csv` exist.

**Two stages, kept strictly apart (rule 3):**
1. **Cross-validation** on the training split ranks all 41 (technique, classifier) pairs → Table 2, Figure 8, Table 4.
2. **Final evaluation**: the top 5 pairs by CV AUC-PR plus the best untreated baseline are refitted on the full
   training split. Thresholds and operating points are chosen on **validation** and applied unchanged to **test**.
   Test is used once, here, and nothing is selected by looking at it.

In [ ]:
import logging
import sys

sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s")

import pandas as pd
from IPython.display import display

from src import config
from src import results_analysis as ra
from src.data_loader import FraudDataset

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 60)

## 5.2 Technique comparison — Table 2

Mean over the 5 CV folds, sorted by AUC-PR (primary metric, rule 2). `f1`, `mcc` and `precision_at_100` use a
threshold chosen on the same fold, so they are optimistic; AUC-PR is threshold-free and unaffected. Accuracy is
deliberately absent.

In [ ]:
results = ra.load_results()
table2 = ra.cv_summary(results)
ra.save_table(table2, "table2_cv_results")
display(table2)

## 5.4 Latency — Figure 8 and Table 3

**Figure 8** is the novel-contribution plot: CV AUC-PR against single-record p99 latency, with the 100 ms budget.
Upper-left = accurate *and* deployable. It plots AUC-PR, not accuracy — the "accuracy vs latency" label in the plan
would contradict rule 2.

Colour = kind of treatment, marker = classifier, error bars = CV std; hover for exact pairs. Things to look for
when writing §5.4: whether data-level techniques sit at the same latency as their untreated classifier (the
hypothesis), how far right the ensembles sit, and whether anything crosses the budget.

In [ ]:
latency = ra.load_latency()
table3 = ra.latency_table(latency, results)
ra.save_table(table3, "table3_latency")

fig8 = ra.fig_auc_pr_vs_latency(table3)
ra.save_figure(fig8, "figure8_auc_pr_vs_latency")
fig8.show()
display(table3)

Per-record cost (ms) by batch size: how much batching amortises the per-call overhead. Relevant to the API's
batch endpoint and to bulk re-scoring, not to the real-time budget.

In [ ]:
table3b = ra.throughput_table(latency)
ra.save_table(table3b.reset_index(), "table3b_per_record_ms_by_batch")
display(table3b)

## 5.3 Statistical significance — Friedman–Nemenyi (Table 4)

Two comparisons:
- **(a) Data-level techniques**, blocked by classifier × fold (4 × 5 = 20 blocks): does a resampling technique help
  whatever the classifier? `differs_from_reference` flags a significant difference from no resampling.
- **(b) Within each base classifier**, every technique available for it (including its algorithm-level variants)
  over 5 folds, against that classifier's untreated model.

Caveats to state in §5.3: the blocks are CV folds, which share training data, so the tests are somewhat liberal
(Dietterich 1998); and 5 folds give little power, so "not significant" is weak evidence of equivalence.
Algorithm-level techniques outside the four base classifiers (RUSBoost, EasyEnsemble, LightGBM) have no untreated
counterpart, so they appear only in Table 2 and Figure 8.

In [ ]:
data_level = ra.rank_comparison(
    ra.data_level_scores(results),
    "Data-level techniques (classifier × fold)",
    reference="none",
)
ra.save_table(data_level.table, "table4a_ranks_data_level")
display(data_level.table)
fig = ra.fig_mean_ranks(data_level)
ra.save_figure(fig, "figure_ranks_data_level")
fig.show()

In [ ]:
families = ra.family_rankings(results)
table4b = pd.concat(
    [r.table.assign(classifier=clf, friedman_p=r.p_value, cd=r.critical_difference)
     for clf, r in families.items()],
    ignore_index=True,
)
ra.save_table(table4b, "table4b_ranks_by_classifier")
display(table4b)
for clf, r in families.items():
    fig = ra.fig_mean_ranks(r)
    ra.save_figure(fig, f"figure_ranks_{clf}")
    fig.show()

## 5.5 Final models on the test split (Table 5)

The top 5 pairs by CV AUC-PR, plus the best untreated baseline as the reference to beat. Each is refitted on the
whole training split (resampling the training data only), its max-F1 threshold is chosen on validation, and it is
scored once on test.

`auc_pr_lo/hi` is a stratified bootstrap 95% CI. `delta_vs_selected` is the difference from the CV-selected best
model on the **same** bootstrap rows, with its own CI; `significant` means that interval excludes zero. Overlapping
individual CIs do *not* by themselves mean "no difference" — the paired interval is the right test.

Refitting takes a few minutes if a random forest is among them.

In [ ]:
splits, manifest = FraudDataset.load_splits(config.DATA_PROCESSED)

pairs = ra.top_pairs(results)
reference_pair = ra.best_untreated(results)
if reference_pair not in pairs:
    pairs.append(reference_pair)
reference = ra.pair_label(*reference_pair)

finals = ra.fit_final_models(pairs, splits)
table5 = ra.test_table(finals, splits)
ra.save_table(table5, "table5_test_final_models")
display(table5)

In [ ]:
fig = ra.fig_pr_curves(finals, splits.y_test, table5, reference=reference)
ra.save_figure(fig, "figure_pr_curves_test")
fig.show()

In [ ]:
fig = ra.fig_confusion_matrices(finals, splits.y_test)
ra.save_figure(fig, "figure_confusion_matrices_test")
fig.show()

## Operating points and error analysis (Table 6)

For the CV-selected model (first row of Table 5). Each row is a different business rule for the threshold, chosen on
validation and applied to test: max F1; minimum expected cost (£100 per missed fraud, £5 per false alarm — the
indicative costs from config, sensitivity-tested in Ch.6); at least 90% precision; and a fixed analyst capacity of
100 alerts/day (the dataset covers 48 hours, so a day ≈ 142k transactions). `alerts_per_day` scales the test alerts
to a full day.

The threshold is a deployment parameter, not a property of the model: this table is the menu an operations team
would choose from, which is the argument for exposing it in the API.

In [ ]:
selected = finals[0]
table6 = ra.operating_points(selected, splits)
ra.save_table(table6, "table6_operating_points")
print("Model:", selected.label)
display(table6)

False positives vs false negatives across thresholds, on test. Dotted lines are the Table 6 operating points,
all fixed on validation beforehand — the curves are descriptive and no threshold is read off them. Cost is a separate
chart rather than a second y-axis.

In [ ]:
sweep = ra.threshold_sweep(splits.y_test, selected.p_test)
fig = ra.fig_errors_by_threshold(sweep, table6)
ra.save_figure(fig, "figure_errors_by_threshold")
fig.show()
fig = ra.fig_cost_by_threshold(sweep, table6)
ra.save_figure(fig, "figure_cost_by_threshold")
fig.show()

## Feature attribution (SHAP, scope-limited)

Mean |SHAP| for the selected model, on validation rows (every fraud plus a 2,000-row legitimate sample), so test
stays reserved for performance. **Interpretation is limited:** V1–V28 are anonymised PCA components, so "V14
matters most" cannot be translated into a real transaction attribute. Say so plainly in Ch.6 §6.4 rather than
over-claiming explainability.

In [ ]:
importance = ra.shap_importance(selected, splits, manifest["feature_names"])
ra.save_table(importance, "table7_shap_importance")
fig = ra.fig_shap(importance, selected.label)
ra.save_figure(fig, "figure_shap_importance")
fig.show()

## Notes for writing Chapter 5

- **CV threshold-dependent metrics are optimistic** (threshold chosen on the scored fold). Quote F1/MCC/precision
  from Table 5 (test, validation-chosen thresholds); quote AUC-PR from either.
- **LightGBM (`lgbm_is_unbalance`)**: with library defaults at 1:578 its scores collapse onto a few distinct values
  saturating at 1.0, so AUC-PR is near the no-skill level while ROC-AUC looks respectable. This is a
  property of the defaults under this imbalance, not of class weighting alone, and there is no untreated LightGBM in
  the grid to separate the two. Report it as such rather than as "LightGBM is a bad technique".
- **RUSBoost** uses imblearn's default base learner, a depth-1 stump; Seiffert et al. (2010) used full C4.5 trees.
  Its weak score should be read with that deviation in mind.
- **SMOTE-Tomek / SMOTE-ENN** were re-run after a fix: the first sweep never cleaned either class, so those rows were
  plain SMOTE. Check the `n_train` column in Table 2 differs from SMOTE's before quoting them.
- **Friedman–Nemenyi** blocks are CV folds, not independent datasets: liberal and low-powered (see §5.3 above).